# Methodological comparison across variants

Reads `results/<variant>/{real,null1,null2}/metrics.json` and produces the decision-surface table + figures consolidating across variants.

Run each variant first (from its worktree or from main):
```bash
python -m pls_explorer.runner --config configs/<name>.yaml
```

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pls_explorer.config import RESULTS_DIR

In [2]:
rows = []
for variant_dir in sorted(RESULTS_DIR.iterdir()):
    if not variant_dir.is_dir():
        continue
    for label in ['real', 'null1', 'null2']:
        mfile = variant_dir / label / 'metrics.json'
        if not mfile.exists():
            continue
        with open(mfile) as f:
            m = json.load(f)
        rows.append({
            'variant': variant_dir.name,
            'label': label,
            'n_comp': m['n_components'],
            'q2': m['q2_global'],
            'rmsecv': m['rmsecv'],
            'rmsecv_over_rmsec': m['rmsecv_over_rmsec'],
            'diagonality': m['w_structure']['diagonality_index'],
            'off_diag_entropy': m['w_structure']['off_diagonal_entropy'],
            'perm_p': m.get('permutation_p_value', np.nan),
        })
df = pd.DataFrame(rows)
df

,variant,label,n_comp,q2,rmsecv,rmsecv_over_rmsec,diagonality,off_diag_entropy,perm_p
0,autoscaled,real,7,0.267699,1.083965,1.237624,0.014749,0.887015,0.0
1,autoscaled,null1,7,0.244212,0.754981,1.260436,0.029808,0.831993,0.0
2,autoscaled,null2,7,0.245504,0.673402,1.268822,0.003186,0.845966,0.0
3,baseline,real,8,0.272048,1.080742,1.268715,0.018123,0.859703,0.0
4,baseline,null1,8,0.292425,0.730504,1.284605,0.049353,0.799793,0.0
5,baseline,null2,8,0.287188,0.654536,1.306015,0.003164,0.807201,0.0
6,pareto,real,8,0.283563,1.072160,1.267199,0.017309,0.873537,0.0
7,pareto,null1,8,0.280531,0.736618,1.291199,0.041106,0.816302,0.0
8,pareto,null2,8,0.281230,0.657265,1.305063,0.003086,0.823027,0.0
9,row_normalized,real,8,0.238822,0.039344,1.213058,0.025611,0.838794,0.0


## Decision-surface table

Pivot to show variants × {real, null1, null2} for each key metric.

In [3]:
for metric in ['q2', 'rmsecv', 'diagonality', 'off_diag_entropy']:
    print(f'\n=== {metric} ===')
    print(df.pivot(index='variant', columns='label', values=metric).round(3))


=== q2 ===
label           null1  null2   real
variant                            
autoscaled      0.244  0.246  0.268
baseline        0.292  0.287  0.272
pareto          0.281  0.281  0.284
row_normalized  0.195  0.240  0.239

=== rmsecv ===
label           null1  null2   real
variant                            
autoscaled      0.755  0.673  1.084
baseline        0.731  0.655  1.081
pareto          0.737  0.657  1.072
row_normalized  0.028  0.044  0.039

=== diagonality ===
label           null1  null2   real
variant                            
autoscaled      0.030  0.003  0.015
baseline        0.049  0.003  0.018
pareto          0.041  0.003  0.017
row_normalized  0.059  0.003  0.026

=== off_diag_entropy ===
label           null1  null2   real
variant                            
autoscaled      0.832  0.846  0.887
baseline        0.800  0.807  0.860
pareto          0.816  0.823  0.874
row_normalized  0.778  0.760  0.839


## Real-vs-nulls discrimination

Which variant gives the cleanest separation between real and each null?

In [4]:
wide = df.pivot(index='variant', columns='label', values='diagonality')
wide['real_minus_null1'] = wide['real'] - wide['null1']
wide['real_minus_null2'] = wide['real'] - wide['null2']
wide['span'] = wide[['real', 'null1', 'null2']].max(axis=1) - wide[['real', 'null1', 'null2']].min(axis=1)
wide.sort_values('span', ascending=False)

label,null1,null2,real,real_minus_null1,real_minus_null2,span
variant,,,,,,
row_normalized,0.058799,0.002836,0.025611,-0.033188,0.022775,0.055963
baseline,0.049353,0.003164,0.018123,-0.031230,0.014959,0.046189
pareto,0.041106,0.003086,0.017309,-0.023797,0.014223,0.038020
autoscaled,0.029808,0.003186,0.014749,-0.015059,0.011563,0.026622
